# Data Dharma by Srikanth
## SQL ↔ PySpark Bridge — Part 4
### JOINs: INNER, LEFT, RIGHT, FULL OUTER

**PART 1 — FOUNDATIONS ✅**
SELECT, FILTER, DISTINCT, SORT, LIMIT

**PART 2 — TRANSFORMATIONS ✅**
CASE WHEN, CAST, IN, LIKE, NULL, String & Date Functions

**PART 3 — AGGREGATIONS ✅**
GROUP BY, COUNT, SUM, AVG, MIN, MAX, HAVING

**PART 4 — JOINS 🚀**
INNER, LEFT, RIGHT, FULL OUTER JOIN

## SECTION 1 — WHY DO WE NEED JOINs?

`orders` contains information about transactions. `customers` contains information about customers.

Suppose the business asks: **"Show each order together with the customer's name."**

`orders` alone can't fully answer that — it only has `customer_id`, not the customer's name or segment. `customers` alone can't answer it either — it has no orders. We need to combine related rows from both tables.

```
ORDERS
      \
       customer_id
      /
CUSTOMERS
       ↓
COMBINED RESULT
```

**Memory:** `JOIN` = combine related rows from two tables.

## SECTION 2 — UNDERSTAND THE MATCHING KEY

Before learning JOIN types, here's the idea in its simplest form — using a small subset of our actual dataset:

```
ORDERS                    CUSTOMERS

customer_id                customer_id
    501   → order            501   → Raj Kumar
    502   → order            502   → Meena Reddy
    512   → order

501 → in BOTH tables
502 → in BOTH tables
512 → ORDERS ONLY     (no matching customer record)
                          515 → CUSTOMERS ONLY  (no order yet)
```

This is a simplified subset used to explain JOIN behavior — the full dataset has more matching customer IDs, and several customers place more than one order. But the shape is exactly this: most customers match, one customer exists with no orders, and one order references a customer that isn't in the `customers` table. We'll use these same real IDs — `501`, `502`, `512`, `515` — to make every JOIN type below show a *visibly different* result.

The JOIN condition is what tells Spark how rows relate:

```
orders.customer_id = customers.customer_id
```

**Memory:** JOIN CONDITION → HOW DO ROWS MATCH?

## SECTION 0 — Data Setup

Reusing the exact same `orders` dataset from Parts 1–3 — same rows, same `order_amount` formula. This notebook is independently runnable — you don't need to run earlier parts first.

We're adding a new `customers` table. It's built so the matched/unmatched pattern from Section 2 is real, not just illustrated:

- **13 customers match an order** (most of the dataset)
- **Customer `515` (Anjali Menon) has no orders yet** — customers-only
- **Order `1014` (customer `512`, Sneha Gupta) has no matching customer record** — orders-only

`customers` also has its own `customer_name` and `city` columns — on purpose. That sets up an important lesson later in Section 11.

In [0]:
# from datetime is a module in Python and date is a class in the datetime module
from datetime import date

# from pyspark.sql.types is a module in Python and StructType is a class in it
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DoubleType,
)

# from pyspark.sql.functions is a module in Python
# each of these is a function in the pyspark.sql.functions module
from pyspark.sql.functions import (
    col,
    coalesce,
    round as spark_round,
)

In [0]:
# below is the schema for the orders table
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("ship_date", DateType(), True),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_amount", DoubleType(), False),
    StructField("promo_code", StringType(), True),
])

In [0]:
# below is the list with sample data
orders_data = [
    (1001, 501, "  raj kumar",      date(2026, 1, 5),  date(2026, 1, 8),  "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 3, 250.00, 20.00, "SAVE10"),
    (1002, 502, "MEENA REDDY ",     date(2026, 1, 6),  date(2026, 1, 8),  "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  2, 150.00, 10.00, None),
    (1003, 503, "Suresh Babu",      date(2026, 1, 7),  None,              "Austin",   "TX", "PENDING",   "PAYPAL",      1, 500.00,  0.00, "WELCOME5"),
    (1004, 504, "Anita Rao",        date(2026, 1, 8),  date(2026, 1, 12), "Chicago",  "IL", "COMPLETED", "UPI",         5,  80.00, 15.00, None),
    (1005, 505, "Kiran Varma",      date(2026, 1, 9),  None,              "New York", "NY", "CANCELLED", "CREDIT_CARD", 4, 120.00,  0.00, None),
    (1006, 501, " Raj Kumar",       date(2026, 1, 10), date(2026, 1, 13), "Houston",  "TX", "COMPLETED", "CREDIT_CARD", 2, 300.00, 25.00, "SAVE10"),
    (1007, 506, "Divya Nair",       date(2026, 1, 11), date(2026, 1, 13), "Dallas",   "TX", "RETURNED",  "DEBIT_CARD",  1, 200.00,  0.00, None),
    (1008, 507, "ramesh iyer",      date(2026, 1, 12), date(2026, 1, 17), "Austin",   "TX", "COMPLETED", "PAYPAL",      3, 100.00, 10.00, "FESTIVE20"),
    (1009, 508, "Priya Sharma",     date(2026, 1, 13), None,              "Chicago",  "IL", "PENDING",   "UPI",         2,  60.00,  5.00, None),
    (1010, 509, "Arjun Menon",      date(2026, 1, 14), date(2026, 1, 17), "New York", "NY", "COMPLETED", "CREDIT_CARD", 6,  90.00, 20.00, "WELCOME5"),
    (1011, 502, "MEENA REDDY",      date(2026, 1, 15), date(2026, 1, 17), "Dallas",   "TX", "COMPLETED", "DEBIT_CARD",  4, 175.00, 30.00, None),
    (1012, 510, "Lakshmi Pillai",   date(2026, 1, 16), date(2026, 1, 22), "Houston",  "TX", "COMPLETED", "PAYPAL",      1, 800.00, 50.00, "SAVE10"),
    (1013, 511, "Vikram Rao",       date(2026, 1, 17), None,              "Austin",   "TX", "CANCELLED", "UPI",         2, 250.00,  0.00, None),
    (1014, 512, "Sneha Gupta",      date(2026, 1, 18), date(2026, 1, 22), "Chicago",  "IL", "COMPLETED", "CREDIT_CARD", 3, 220.00, 10.00, "FESTIVE20"),
    (1015, 513, "Karthik Reddy",    date(2026, 1, 19), date(2026, 1, 22), "New York", "NY", "RETURNED",  "DEBIT_CARD",  1, 400.00,  0.00, None),
    (1016, 503, "  Suresh Babu  ",  date(2026, 1, 20), date(2026, 1, 22), "Austin",   "TX", "COMPLETED", "PAYPAL",      5,  60.00,  5.00, "WELCOME5"),
    (1017, 514, "Deepa Krishnan",   date(2026, 1, 21), None,              "Houston",  "TX", "PENDING",   "CREDIT_CARD", 2, 175.00,  0.00, None),
    (1018, 505, "Kiran Varma",      date(2026, 1, 22), date(2026, 1, 25), "New York", "NY", "COMPLETED", "UPI",         3, 210.00, 15.00, "SAVE10"),
]

# create a dataframe with the schema and data
orders_raw_df = spark.createDataFrame(orders_data, schema=orders_schema)

In [0]:
# create a new column 'order_amount' by multiplying 'quantity' and 'unit_price' and subtracting 'discount_amount'
# round the result to 2 decimal places and cast it to decimal(10,2)

orders_df = orders_raw_df.withColumn(
    "order_amount",
    spark_round((col("quantity") * col("unit_price")) - col("discount_amount"), 2).cast("decimal(10,2)")
)

# display the dataframe
display(orders_df)

order_id,customer_id,customer_name,order_date,ship_date,city,state,order_status,payment_method,quantity,unit_price,discount_amount,promo_code,order_amount
1001,501,raj kumar,2026-01-05,2026-01-08,Houston,TX,COMPLETED,CREDIT_CARD,3,250.0,20.0,SAVE10,730.00
1002,502,MEENA REDDY,2026-01-06,2026-01-08,Dallas,TX,COMPLETED,DEBIT_CARD,2,150.0,10.0,null,290.00
1003,503,Suresh Babu,2026-01-07,null,Austin,TX,PENDING,PAYPAL,1,500.0,0.0,WELCOME5,500.00
1004,504,Anita Rao,2026-01-08,2026-01-12,Chicago,IL,COMPLETED,UPI,5,80.0,15.0,null,385.00
1005,505,Kiran Varma,2026-01-09,null,New York,NY,CANCELLED,CREDIT_CARD,4,120.0,0.0,null,480.00
1006,501,Raj Kumar,2026-01-10,2026-01-13,Houston,TX,COMPLETED,CREDIT_CARD,2,300.0,25.0,SAVE10,575.00
1007,506,Divya Nair,2026-01-11,2026-01-13,Dallas,TX,RETURNED,DEBIT_CARD,1,200.0,0.0,null,200.00
1008,507,ramesh iyer,2026-01-12,2026-01-17,Austin,TX,COMPLETED,PAYPAL,3,100.0,10.0,FESTIVE20,290.00
1009,508,Priya Sharma,2026-01-13,null,Chicago,IL,PENDING,UPI,2,60.0,5.0,null,115.00
1010,509,Arjun Menon,2026-01-14,2026-01-17,New York,NY,COMPLETED,CREDIT_CARD,6,90.0,20.0,WELCOME5,520.00


In [0]:
# create a temp view 'orders' from the dataframe
orders_df.createOrReplaceTempView("orders")

Now the new `customers` table — 13 customers who placed orders, plus `515` (Anjali Menon), who hasn't ordered anything yet. Notice `512` (Sneha Gupta) is missing on purpose — she placed order `1014`, but has no customer record here.

In [0]:
# below is the schema for the customers table
customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("city", StringType(), False),
])

In [0]:
# below is the list with sample customer data
customers_data = [
    (501, "Raj Kumar",      "Regular", "Houston"),
    (502, "Meena Reddy",    "Premium", "Dallas"),
    (503, "Suresh Babu",    "Regular", "Austin"),
    (504, "Anita Rao",      "New",     "Chicago"),
    (505, "Kiran Varma",    "Premium", "New York"),
    (506, "Divya Nair",     "Regular", "Dallas"),
    (507, "Ramesh Iyer",    "Regular", "Austin"),
    (508, "Priya Sharma",   "New",     "Chicago"),
    (509, "Arjun Menon",    "Regular", "New York"),
    (510, "Lakshmi Pillai", "Premium", "Houston"),
    (511, "Vikram Rao",     "Regular", "Austin"),
    (513, "Karthik Reddy",  "New",     "New York"),
    (514, "Deepa Krishnan", "Regular", "Houston"),
    (515, "Anjali Menon",   "New",     "Houston"),
]

# create a dataframe with the schema and data
customers_df = spark.createDataFrame(customers_data, schema=customers_schema)

# display the dataframe
display(customers_df)

customer_id,customer_name,customer_segment,city
501,Raj Kumar,Regular,Houston
502,Meena Reddy,Premium,Dallas
503,Suresh Babu,Regular,Austin
504,Anita Rao,New,Chicago
505,Kiran Varma,Premium,New York
506,Divya Nair,Regular,Dallas
507,Ramesh Iyer,Regular,Austin
508,Priya Sharma,New,Chicago
509,Arjun Menon,Regular,New York
510,Lakshmi Pillai,Premium,Houston


In [0]:
# create a temp view 'customers' from the dataframe
customers_df.createOrReplaceTempView("customers")

From this point onward, SQL and PySpark are reading the same two tables —
SQL through the views `orders` and `customers`, PySpark through the DataFrames `orders_df` and `customers_df`.

## SECTION 3 — INNER JOIN

**Business Requirement:** Show only orders that have a matching customer.

**Memory:** `INNER` → matched rows only, from **both** sides.

### 🟨 SQL

In [0]:
%sql
-- Show orders with matching customer details

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.city AS order_city
FROM orders o
INNER JOIN customers c 
  ON o.customer_id = c.customer_id
ORDER BY o.order_id;

### 🟦 PySpark

In [0]:
# Show orders with matching customer details

o = orders_df.alias("o")
c = customers_df.alias("c")

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "inner")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_name"),
        col("o.city").alias("order_city")
    )
    .orderBy("order_id")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_city
1001,501,Raj Kumar,Houston
1002,502,Meena Reddy,Dallas
1003,503,Suresh Babu,Austin
1004,504,Anita Rao,Chicago
1005,505,Kiran Varma,New York
1006,501,Raj Kumar,Houston
1007,506,Divya Nair,Dallas
1008,507,Ramesh Iyer,Austin
1009,508,Priya Sharma,Chicago
1010,509,Arjun Menon,New York


**Key Mapping**

`INNER JOIN` ↔ `.join(..., "inner")`

17 rows come back — every order **except** `1014`, whose customer (`512`) has no record in `customers`.

## SECTION 4 — LEFT JOIN

**Business Requirement:** Keep ALL orders and add customer details when available.

**Memory:** `LEFT` → keep everything on the left, add matches from the right when they exist.

### 🟨 SQL

In [0]:
%sql
-- Keep all orders and add customer details when available

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.city AS order_city
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id;

order_id,customer_id,customer_name,order_city
1001,501,Raj Kumar,Houston
1002,502,Meena Reddy,Dallas
1003,503,Suresh Babu,Austin
1004,504,Anita Rao,Chicago
1005,505,Kiran Varma,New York
1006,501,Raj Kumar,Houston
1007,506,Divya Nair,Dallas
1008,507,Ramesh Iyer,Austin
1009,508,Priya Sharma,Chicago
1010,509,Arjun Menon,New York


### 🟦 PySpark

In [0]:
# Keep all orders and add customer details when available

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "left")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_name"),
        col("o.city").alias("order_city")
    )
    .orderBy("order_id")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_city
1001,501,Raj Kumar,Houston
1002,502,Meena Reddy,Dallas
1003,503,Suresh Babu,Austin
1004,504,Anita Rao,Chicago
1005,505,Kiran Varma,New York
1006,501,Raj Kumar,Houston
1007,506,Divya Nair,Dallas
1008,507,Ramesh Iyer,Austin
1009,508,Priya Sharma,Chicago
1010,509,Arjun Menon,New York


**Key Mapping**

`LEFT JOIN` ↔ `.join(..., "left")`

All 18 orders come back. Look at `order_id = 1014` — `customer_name` is `NULL`, because customer `512` doesn't exist in `customers`. The order didn't disappear; it just has no customer details to show.

**Memory:** NO MATCH ≠ ROW DISAPPEARS. Whether the row survives depends on the JOIN TYPE.

## SECTION 5 — RIGHT JOIN

**Business Requirement:** Keep ALL customers and add order details when available.

**Memory:** `RIGHT` → keep everything on the right, add matches from the left when they exist.

### 🟨 SQL

In [0]:
%sql
-- Keep all customers and add order details when available

SELECT
    o.order_id,
    o.city AS order_city,
    c.customer_id,
    c.customer_name

FROM orders o
RIGHT JOIN customers c ON o.customer_id = c.customer_id
ORDER BY c.customer_id;

order_id,order_city,customer_id,customer_name
1006,Houston,501,Raj Kumar
1001,Houston,501,Raj Kumar
1011,Dallas,502,Meena Reddy
1002,Dallas,502,Meena Reddy
1016,Austin,503,Suresh Babu
1003,Austin,503,Suresh Babu
1004,Chicago,504,Anita Rao
1018,New York,505,Kiran Varma
1005,New York,505,Kiran Varma
1007,Dallas,506,Divya Nair


### 🟦 PySpark

In [0]:
# Keep all customers and add order details when available

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "right")
    .select(
        col("o.order_id"),
        col("o.city").alias("order_city"),
        col("c.customer_id"),
        col("c.customer_name")
    )
    .orderBy("customer_id")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_city,customer_id,customer_name
1006,Houston,501,Raj Kumar
1001,Houston,501,Raj Kumar
1011,Dallas,502,Meena Reddy
1002,Dallas,502,Meena Reddy
1016,Austin,503,Suresh Babu
1003,Austin,503,Suresh Babu
1004,Chicago,504,Anita Rao
1018,New York,505,Kiran Varma
1005,New York,505,Kiran Varma
1007,Dallas,506,Divya Nair


**Key Mapping**

`RIGHT JOIN` ↔ `.join(..., "right")`

All 14 customers are preserved. The result contains 18 rows because some customers have multiple matching orders — customers `501`, `502`, `503`, and `505` each placed more than one order, so each of them appears more than once in the joined result. A `JOIN` can produce multiple rows for one customer when that customer matches multiple orders — **JOIN output rows are not the same thing as the number of customers.**

Look at `customer_id = 515` — `order_id` is `NULL`, because Anjali Menon hasn't placed an order yet.

## SECTION 6 — FULL OUTER JOIN

**Business Requirement:** Show everything from both Orders and Customers, whether matched or unmatched.

**Memory:** `FULL` → keep everything from both sides.

### 🟨 SQL

In [0]:
%sql
-- Show every order and every customer, matched or not

SELECT
    COALESCE(o.customer_id, c.customer_id) AS customer_id,
    o.order_id,
    c.customer_name
FROM orders o
FULL OUTER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY customer_id, o.order_id;

customer_id,order_id,customer_name
501,1001,Raj Kumar
501,1006,Raj Kumar
502,1002,Meena Reddy
502,1011,Meena Reddy
503,1003,Suresh Babu
503,1016,Suresh Babu
504,1004,Anita Rao
505,1005,Kiran Varma
505,1018,Kiran Varma
506,1007,Divya Nair


### 🟦 PySpark

In [0]:
# Show every order and every customer, matched or not

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "full")
    .select(
        coalesce(col("o.customer_id"), col("c.customer_id")).alias("customer_id"),
        col("o.order_id"),
        col("c.customer_name")
    )
    .orderBy("customer_id", "order_id")
)

# display data in the DataFrame result_df
display(result_df)

customer_id,order_id,customer_name
501,1001,Raj Kumar
501,1006,Raj Kumar
502,1002,Meena Reddy
502,1011,Meena Reddy
503,1003,Suresh Babu
503,1016,Suresh Babu
504,1004,Anita Rao
505,1005,Kiran Varma
505,1018,Kiran Varma
506,1007,Divya Nair


**Key Mapping**

`FULL OUTER JOIN` ↔ `.join(..., "full")`

19 rows — everything from Section 3 (17 matched) plus the unmatched order (`1014`) plus the unmatched customer (`515`).

We use `COALESCE` here (from Part 2) because a `FULL OUTER JOIN` gives us **two** `customer_id` columns — `o.customer_id` and `c.customer_id` — and exactly one of them is `NULL` on any unmatched row. `COALESCE` picks whichever one actually has a value, so we get a single clean `customer_id` column to sort and display by.

## SECTION 7 — COMPARE ALL FOUR JOIN TYPES

Using the simple `501 / 502 / 512 / 515` idea from Section 2 (a simplified subset — the full dataset has more matches and multiple orders per customer) — here's what each JOIN type keeps:

```
INNER → 501, 502
LEFT  → 501, 502, 512
RIGHT → 501, 502, 515
FULL  → 501, 502, 512, 515
```

```
INNER → matched only
LEFT  → matched + unmatched from the left
RIGHT → matched + unmatched from the right
FULL  → everything from both sides
```

And here's what actually happened with our real data — same pattern, real row counts:

| JOIN Type | Row Count | What's included |
|---|---|---|
| INNER | 17 | Only orders with a matching customer |
| LEFT | 18 | All orders; order `1014` has `NULL` customer |
| RIGHT | 18 | All customers; customer `515` has `NULL` order |
| FULL | 19 | Everything — matched, order-only, and customer-only |

**Worth noticing:** `LEFT` and `RIGHT` both returned 18 rows — but they're not the same 18 rows! `LEFT` includes order `1014` (with no customer); `RIGHT` includes customer `515` (with no order). Matching row *counts* doesn't mean matching *results* — always check what's actually inside.

**Memory table — the most important takeaway in this notebook:**

| JOIN Type | Memory |
|---|---|
| INNER | BOTH |
| LEFT | KEEP ALL LEFT |
| RIGHT | KEEP ALL RIGHT |
| FULL | KEEP EVERYTHING |

## SECTION 8 — SELECT COLUMNS AFTER JOIN

**Business Requirement:** Show order ID, customer ID, customer name, order status, and order amount.

After joining two DataFrames, we usually don't need every column from both sides.

**Memory:** `JOIN` → combine rows. `SELECT` → keep columns.

### 🟨 SQL

In [0]:
%sql
-- Show order id, customer id, customer name, order status, and order amount

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.order_status,
    o.order_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id;

order_id,customer_id,customer_name,order_status,order_amount
1001,501,Raj Kumar,COMPLETED,730.00
1002,502,Meena Reddy,COMPLETED,290.00
1003,503,Suresh Babu,PENDING,500.00
1004,504,Anita Rao,COMPLETED,385.00
1005,505,Kiran Varma,CANCELLED,480.00
1006,501,Raj Kumar,COMPLETED,575.00
1007,506,Divya Nair,RETURNED,200.00
1008,507,Ramesh Iyer,COMPLETED,290.00
1009,508,Priya Sharma,PENDING,115.00
1010,509,Arjun Menon,COMPLETED,520.00


### 🟦 PySpark

In [0]:
# Show order id, customer id, customer name, order status, and order amount

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "inner")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_name"),
        col("o.order_status"),
        col("o.order_amount")
    )
    .orderBy("order_id")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_status,order_amount
1001,501,Raj Kumar,COMPLETED,730.00
1002,502,Meena Reddy,COMPLETED,290.00
1003,503,Suresh Babu,PENDING,500.00
1004,504,Anita Rao,COMPLETED,385.00
1005,505,Kiran Varma,CANCELLED,480.00
1006,501,Raj Kumar,COMPLETED,575.00
1007,506,Divya Nair,RETURNED,200.00
1008,507,Ramesh Iyer,COMPLETED,290.00
1009,508,Priya Sharma,PENDING,115.00
1010,509,Arjun Menon,COMPLETED,520.00


**Key Mapping**

`SELECT column1, column2, ...` after a `JOIN` ↔ `.select(col("t.column1"), ...)` after a `.join()`

## SECTION 9 — DIFFERENT JOIN KEY NAMES

**Business Requirement:** Join Orders to Customers even though the key column names are different.

Real-world tables don't always use identical column names. Here, `customer_id` in `customers` has simply been renamed `customer_key` — the JOIN condition tells Spark how the columns relate, and the names don't need to match.

In [0]:
# create a small demonstration table where the key column is named differently

customers_alt_df = customers_df.withColumnRenamed("customer_id", "customer_key")
customers_alt_df.createOrReplaceTempView("customers_alt_key")

### 🟨 SQL

In [0]:
%sql
-- Join orders to customers even though the key column names are different

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name
FROM orders o
INNER JOIN customers_alt_key c ON o.customer_id = c.customer_key
ORDER BY o.order_id;

order_id,customer_id,customer_name
1001,501,Raj Kumar
1002,502,Meena Reddy
1003,503,Suresh Babu
1004,504,Anita Rao
1005,505,Kiran Varma
1006,501,Raj Kumar
1007,506,Divya Nair
1008,507,Ramesh Iyer
1009,508,Priya Sharma
1010,509,Arjun Menon


### 🟦 PySpark

In [0]:
# Join orders to customers even though the key column names are different

oa = orders_df.alias("o")
ca = customers_alt_df.alias("c")

result_df = (
    oa.join(ca, col("o.customer_id") == col("c.customer_key"), "inner")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_name")
    )
    .orderBy("order_id")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name
1001,501,Raj Kumar
1002,502,Meena Reddy
1003,503,Suresh Babu
1004,504,Anita Rao
1005,505,Kiran Varma
1006,501,Raj Kumar
1007,506,Divya Nair
1008,507,Ramesh Iyer
1009,508,Priya Sharma
1010,509,Arjun Menon


**Key Mapping**

`ON o.customer_id = c.customer_key` ↔ `col("o.customer_id") == col("c.customer_key")`

Same result as Section 3 — the column names were different, but the JOIN condition still tells Spark exactly how the rows relate.

## SECTION 10 — FILTER AFTER JOIN

**Business Requirement:** Show completed orders together with customer information.

**Memory:** `JOIN` → combine related rows. `FILTER` → keep only rows meeting a condition.

### 🟨 SQL

In [0]:
%sql
-- Show completed orders together with customer information

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.order_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'COMPLETED'
ORDER BY o.order_id;

order_id,customer_id,customer_name,order_amount
1001,501,Raj Kumar,730.00
1002,502,Meena Reddy,290.00
1004,504,Anita Rao,385.00
1006,501,Raj Kumar,575.00
1008,507,Ramesh Iyer,290.00
1010,509,Arjun Menon,520.00
1011,502,Meena Reddy,670.00
1012,510,Lakshmi Pillai,750.00
1016,503,Suresh Babu,295.00
1018,505,Kiran Varma,615.00


### 🟦 PySpark

In [0]:
# Show completed orders together with customer information

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "inner")
    .filter(col("o.order_status") == "COMPLETED")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_name"),
        col("o.order_amount")
    )
    .orderBy("order_id")
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,order_amount
1001,501,Raj Kumar,730.00
1002,502,Meena Reddy,290.00
1004,504,Anita Rao,385.00
1006,501,Raj Kumar,575.00
1008,507,Ramesh Iyer,290.00
1010,509,Arjun Menon,520.00
1011,502,Meena Reddy,670.00
1012,510,Lakshmi Pillai,750.00
1016,503,Suresh Babu,295.00
1018,505,Kiran Varma,615.00


**Key Mapping**

`WHERE` after a `JOIN` ↔ `.filter()` after a `.join()`

10 rows — 11 completed orders minus order `1014`, which the `INNER JOIN` already dropped because customer `512` doesn't exist in `customers`.

## SECTION 11 — DUPLICATE / AMBIGUOUS COLUMN NAMES

Both `orders` and `customers` have a `city` column — `orders.city` is where the order shipped, `customers.city` is where the customer lives. After a JOIN, writing `col("city")` alone would be ambiguous — PySpark wouldn't know which one you mean.

That's exactly why every example in this notebook has used `.alias("o")` / `.alias("c")` and qualified references like `col("o.city")` / `col("c.city")` from the start.

**Memory:**
`o.city` → city from ORDERS
`c.city` → city from CUSTOMERS

**Business Requirement:** Show each order's shipping city alongside the customer's home city.

### 🟨 SQL

In [0]:
%sql
-- Show each order's shipping city alongside the customer's home city

SELECT
    o.order_id,
    o.city AS order_city,
    c.city AS customer_home_city
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id;

order_id,order_city,customer_home_city
1001,Houston,Houston
1002,Dallas,Dallas
1003,Austin,Austin
1004,Chicago,Chicago
1005,New York,New York
1006,Houston,Houston
1007,Dallas,Dallas
1008,Austin,Austin
1009,Chicago,Chicago
1010,New York,New York


### 🟦 PySpark

In [0]:
# Show each order's shipping city alongside the customer's home city

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "inner")
    .select(
        col("o.order_id"),
        col("o.city").alias("order_city"),
        col("c.city").alias("customer_home_city")
    )
    .orderBy("order_id")
)

# display data in the DataFrame result_df
display(result_df)

order_id,order_city,customer_home_city
1001,Houston,Houston
1002,Dallas,Dallas
1003,Austin,Austin
1004,Chicago,Chicago
1005,New York,New York
1006,Houston,Houston
1007,Dallas,Dallas
1008,Austin,Austin
1009,Chicago,Chicago
1010,New York,New York


**Key Mapping**

`alias()` gives each DataFrame a short name. `o.city` / `c.city` — the qualified reference — tells PySpark exactly which table's column you mean, the same way `o.city` and `c.city` work in the SQL `ON`/`SELECT` clauses.

## SECTION 12 — COMBINED REAL-WORLD EXAMPLE

**Business Requirement:** Create a completed-order customer report showing order ID, customer ID, customer name, customer segment, order date, order status, order amount, and customer city. Keep all completed orders even if customer details are missing. Sort highest order amount first.

### 🟨 SQL

In [0]:
%sql
-- Build a completed-order customer report, keeping orders even without a customer match

SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    c.customer_segment,
    o.order_date,
    o.order_status,
    o.order_amount,
    c.city AS customer_city
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'COMPLETED'
ORDER BY o.order_amount DESC, o.order_id ASC;

order_id,customer_id,customer_name,customer_segment,order_date,order_status,order_amount,customer_city
1012,510,Lakshmi Pillai,Premium,2026-01-16,COMPLETED,750.00,Houston
1001,501,Raj Kumar,Regular,2026-01-05,COMPLETED,730.00,Houston
1011,502,Meena Reddy,Premium,2026-01-15,COMPLETED,670.00,Dallas
1014,512,null,null,2026-01-18,COMPLETED,650.00,null
1018,505,Kiran Varma,Premium,2026-01-22,COMPLETED,615.00,New York
1006,501,Raj Kumar,Regular,2026-01-10,COMPLETED,575.00,Houston
1010,509,Arjun Menon,Regular,2026-01-14,COMPLETED,520.00,New York
1004,504,Anita Rao,New,2026-01-08,COMPLETED,385.00,Chicago
1016,503,Suresh Babu,Regular,2026-01-20,COMPLETED,295.00,Austin
1002,502,Meena Reddy,Premium,2026-01-06,COMPLETED,290.00,Dallas


### 🟦 PySpark

In [0]:
# Build a completed-order customer report, keeping orders even without a customer match

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "left")
    .filter(col("o.order_status") == "COMPLETED")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_name"),
        col("c.customer_segment"),
        col("o.order_date"),
        col("o.order_status"),
        col("o.order_amount"),
        col("c.city").alias("customer_city")
    )
    .orderBy(col("o.order_amount").desc(), col("o.order_id").asc())
)

# display data in the DataFrame result_df
display(result_df)

order_id,customer_id,customer_name,customer_segment,order_date,order_status,order_amount,customer_city
1012,510,Lakshmi Pillai,Premium,2026-01-16,COMPLETED,750.00,Houston
1001,501,Raj Kumar,Regular,2026-01-05,COMPLETED,730.00,Houston
1011,502,Meena Reddy,Premium,2026-01-15,COMPLETED,670.00,Dallas
1014,512,null,null,2026-01-18,COMPLETED,650.00,null
1018,505,Kiran Varma,Premium,2026-01-22,COMPLETED,615.00,New York
1006,501,Raj Kumar,Regular,2026-01-10,COMPLETED,575.00,Houston
1010,509,Arjun Menon,Regular,2026-01-14,COMPLETED,520.00,New York
1004,504,Anita Rao,New,2026-01-08,COMPLETED,385.00,Chicago
1016,503,Suresh Babu,Regular,2026-01-20,COMPLETED,295.00,Austin
1002,502,Meena Reddy,Premium,2026-01-06,COMPLETED,290.00,Dallas


Same requirement. Same join → filter → select → sort chain. Same result — expressed two ways.

Look for `order_id = 1014` in the results — it's still there (thanks to `LEFT JOIN`), with `customer_name`, `customer_segment`, and `customer_city` all `NULL`. That's exactly the "keep all completed orders even if customer details are missing" requirement, made visible.

## SECTION 13 — VIEWER CHALLENGE

**Pause the video and try this first.**

**Requirement:** Create a report that keeps ALL customers and shows their order details when available.

1. Which JOIN should you use?
2. Write it first in SQL.
3. Translate it into PySpark.
4. Select only useful columns: `customer_id`, `customer_name`, `order_id`, `order_status`, `order_amount`.
5. Sort by `customer_id`.

### 🟨 SQL SOLUTION

In [0]:
%sql
-- Keep all customers and show their order details when available

SELECT
    c.customer_id,
    c.customer_name,
    o.order_id,
    o.order_status,
    o.order_amount
FROM orders o
RIGHT JOIN customers c ON o.customer_id = c.customer_id
ORDER BY c.customer_id;

### 🟦 PYSPARK SOLUTION

In [0]:
# Keep all customers and show their order details when available

result_df = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "right")
    .select(
        col("c.customer_id"),
        col("c.customer_name"),
        col("o.order_id"),
        col("o.order_status"),
        col("o.order_amount")
    )
    .orderBy("customer_id")
)

# display data in the DataFrame result_df
display(result_df)

**RESULT:** `RIGHT JOIN` was the right call — it keeps all 14 customers. Customer `515` (Anjali Menon) appears with `order_id`, `order_status`, and `order_amount` all `NULL`, exactly as expected: no orders yet, but still shown.

## Final Check — Did SQL and PySpark Return the Same JOIN Result?

We wrote the same JOIN logic in SQL and PySpark, using the Section 12 combined example.

Each row has **8** selected business columns — `order_id`, `customer_id`, `customer_name`, `customer_segment`, `order_date`, `order_status`, `order_amount`, and `customer_city`. So we compare the **complete row**, not just one identifier.

### What will we do?

1. Run the SQL version and store the result in `sql_result`
2. Run the PySpark version and store the result in `pyspark_result`
3. Both are already sorted by `order_amount DESC, order_id ASC`, so the comparison is order-sensitive and fair
4. Collect each full row into a Python list
5. Compare the two lists directly

To be precise about what this verifies: it confirms that SQL and PySpark returned the **same JOIN result for these selected business columns** — not that every possible column in both tables matches, since we deliberately selected only the columns the business report needs.

In [0]:
# Compare SQL and PySpark results to verify the JOIN report matches column by column

# Run the SQL query and store the result as a DataFrame
sql_result = spark.sql("""
    SELECT
        o.order_id,
        o.customer_id,
        c.customer_name,
        c.customer_segment,
        o.order_date,
        o.order_status,
        o.order_amount,
        c.city AS customer_city
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'COMPLETED'
    ORDER BY o.order_amount DESC, o.order_id ASC
""")

# Apply the same logic using PySpark and store the result as a DataFrame
pyspark_result = (
    o.join(c, col("o.customer_id") == col("c.customer_id"), "left")
    .filter(col("o.order_status") == "COMPLETED")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("c.customer_name"),
        col("c.customer_segment"),
        col("o.order_date"),
        col("o.order_status"),
        col("o.order_amount"),
        col("c.city").alias("customer_city")
    )
    .orderBy(col("o.order_amount").desc(), col("o.order_id").asc())
)

# Trigger execution with collect(), bring each full row to the driver, as a dictionary of column values
sql_rows = [row.asDict() for row in sql_result.collect()]

# Trigger execution with collect(), bring each full PySpark row to the driver, as a dictionary of column values
pyspark_rows = [row.asDict() for row in pyspark_result.collect()]

# Display both full result sets and check whether every column matches for every row
print("SQL rows:     ", sql_rows)
print("PySpark rows: ", pyspark_rows)
print("Same full rows (all columns, order-sensitive):", sql_rows == pyspark_rows)

SQL rows:      [{'order_id': 1012, 'customer_id': 510, 'customer_name': 'Lakshmi Pillai', 'customer_segment': 'Premium', 'order_date': datetime.date(2026, 1, 16), 'order_status': 'COMPLETED', 'order_amount': Decimal('750.00'), 'customer_city': 'Houston'}, {'order_id': 1001, 'customer_id': 501, 'customer_name': 'Raj Kumar', 'customer_segment': 'Regular', 'order_date': datetime.date(2026, 1, 5), 'order_status': 'COMPLETED', 'order_amount': Decimal('730.00'), 'customer_city': 'Houston'}, {'order_id': 1011, 'customer_id': 502, 'customer_name': 'Meena Reddy', 'customer_segment': 'Premium', 'order_date': datetime.date(2026, 1, 15), 'order_status': 'COMPLETED', 'order_amount': Decimal('670.00'), 'customer_city': 'Dallas'}, {'order_id': 1014, 'customer_id': 512, 'customer_name': None, 'customer_segment': None, 'order_date': datetime.date(2026, 1, 18), 'order_status': 'COMPLETED', 'order_amount': Decimal('650.00'), 'customer_city': None}, {'order_id': 1018, 'customer_id': 505, 'customer_name': 

### Understanding the Result

`row.asDict()` turns one Spark Row into a Python dictionary of `{column_name: value}` — so `sql_rows` and `pyspark_rows` are each a list of dictionaries, one per order.

`sql_rows == pyspark_rows` compares those lists directly. For this to be `True`, every row must appear in the same order, with the same values in all 8 selected columns — including the `NULL` customer fields on order `1014`.

If the result is `True` ✅, our SQL and PySpark JOIN logic produced identical output for the business columns we selected.

## SECTION 15 — FINAL CHEAT SHEET

| SQL | PySpark |
|---|---|
| `INNER JOIN` | `.join(..., "inner")` |
| `LEFT JOIN` | `.join(..., "left")` |
| `RIGHT JOIN` | `.join(..., "right")` |
| `FULL OUTER JOIN` | `.join(..., "full")` |
| `ON` condition | join condition (2nd argument) |
| table alias | `.alias()` |
| `SELECT` columns | `.select()` |
| `WHERE` after JOIN | `.filter()` |
| `ORDER BY` | `.orderBy()` |

**Memory table:**

| JOIN Type | Memory |
|---|---|
| INNER | MATCHED ONLY |
| LEFT | KEEP ALL LEFT |
| RIGHT | KEEP ALL RIGHT |
| FULL | KEEP EVERYTHING |

**The two questions every JOIN answers:**

`JOIN CONDITION` → HOW DO ROWS MATCH?
`JOIN TYPE` → WHICH ROWS SURVIVE?

```
     SQL
      ↕
   PySpark
      ↓
Same Business Logic
```

## Part 4 complete.

Same data. Same business requirement. Different syntax. Same business logic.

**Coming next — Part 5: SQL ↔ PySpark Window Functions**

A preview of what's ahead:
- `ROW_NUMBER`
- `RANK`
- `DENSE_RANK`
- `PARTITION BY`
- `ORDER BY` inside window functions